# Recourse under predictive multiplicity — V6

This is the single entry point for the replication package. All six datasets are in `dataset/`; all implementation and reporting functions are in `utils/`. No earlier experiment results, notebook versions, Google Drive paths, or dataset downloads are required.

**Start here:** use Python 3.11, install `requirements.txt`, open this notebook from the extracted package directory, and choose **Restart Kernel and Run All Cells**. The default is a small **quick functionality run**, not a reproduction of the paper's confidence intervals. All experiment types are enabled; the additional neural MIQCP reference is off in quick mode.

The two multiplicity-burden experiments consume trade-offs generated earlier in this same notebook. Each experiment saves its configuration, data splits, trained models, individual outcomes, optimizer traces, seed statistics, figures, and tables. Mode changes use separate directories.

Gurobi is required for tree experiments; its restricted distribution can handle only small formulations. Larger profiles and the neural MIQCP reference need a suitable full-size license. See the README for setup and the exact mode settings.

**Target audit:** German's stored `y=1` denotes bad credit; Polish's stored label orientation remains unverified. The historical encodings are retained to reproduce the recorded numerical experiments, and both datasets are visibly flagged. Do not interpret those rows as validated favorable recourse. Neither dataset is in the default quick run.


## 1. Locate the package and prepare the environment
Install dependencies once in your Python 3.11 environment: `python -m pip install -r requirements.txt`. If using a new notebook environment, uncomment the `%pip` line below, run it once, and restart the kernel before continuing. Installation is not repeated by Run All.

In [ ]:
# %pip install -r requirements.txt
import os
import sys
from pathlib import Path

# Normally no edit is needed: launch Jupyter from the extracted package directory.
ROOT = Path.cwd().resolve()
if not (ROOT / "RecourseModelMultiplicity-v6.ipynb").is_file():
    candidates = [ROOT / "RecourseModelMultiplicity-v6", *ROOT.parents]
    ROOT = next((p for p in candidates if (p / "RecourseModelMultiplicity-v6.ipynb").is_file()), ROOT)
if not (ROOT / "utils" / "recourse_runtime.py").is_file() or not (ROOT / "dataset" / "manifest.json").is_file():
    raise FileNotFoundError("Set ROOT to the extracted V6 folder containing dataset/, utils/, and this notebook.")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "results" / ".matplotlib"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Package:", ROOT)


## 2. One control cell for the entire notebook
- `quick`: Synthetic + COMPAS, 2 seeds, 3 instances per seed, 10 comparison rounds, 20 differentiable convergence rounds, 5 tree comparison rounds, and 10 tree convergence rounds.
- `intermediate`: larger sample and iteration counts on four datasets.
- `full`: the documented larger replication settings, including the 200-round convex-gap and 100-round tree-gap runs. The historical generalization experiment remains a separate three-seed exploratory protocol.

The mixed trade-offs keep **(2,2,2), (4,4,4), and (8,8,8)** in every mode. Quick mode uses three budget values per dataset, 25 gradient steps per weighted oracle, 40 training epochs, 500 LIME samples, and forests of five depth-two trees.

`RESUME=True` reuses only completed local jobs with matching code, data, packages, and settings. Changing modes does not overwrite previous runs. `INCLUDE_NEURAL_MIQCP=None` follows the selected mode (off for quick; on for larger modes). Set it explicitly to `False` for a non-MIQCP comparison. Setting `RUN_TREES=False` explicitly skips the three tree sections and their burden analysis.

In [ ]:
RUN_MODE = "quick"
SEED = 42
RESUME = True
RUN_TREES = True
INCLUDE_NEURAL_MIQCP = None
SHOW_PLOTS = True

from IPython.display import display
from utils import recourse_runtime as runtime
from utils import recourse_campaign as campaign

inventory, workload = runtime.initialize(
    mode=RUN_MODE, seed=SEED, run_trees=RUN_TREES,
    include_neural_miqcp=INCLUDE_NEURAL_MIQCP, root=ROOT,
)
display(inventory)
display(workload)


## 3. Mixed-model loss convergence
Track worst-model squared-probability loss for mixed differentiable sets. This is a heuristic loss trajectory, not a convex duality-gap guarantee.

In [ ]:
loss_convergence = runtime.run_section(
    "convergence", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
)
display(runtime.compact_summary(loss_convergence))

## 4. Convex duality-gap convergence
Use affine-logit logistic models with normalized binary cross-entropy. Evaluate the averaged recourse point and average pre-update adversary weights. Solver bounds and seed confidence intervals are saved separately.

In [ ]:
duality_gap = runtime.run_section(
    "convergence_duality", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
)
display(runtime.compact_summary(duality_gap))

## 5. Homogeneous logistic models
Compare ORPM, ADV, ElliCE, and ROAR on three competing logistic models. All methods use the same held-out factual cohort and feature constraints.

In [ ]:
linear_results = runtime.run_section(
    "linear_baselines", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
)
display(runtime.compact_summary(linear_results))

## 6. Unseen-model generalization
Optimize recourse against one set and evaluate all-model validity on a separately generated unseen set. Quick uses 3 optimization and 5 unseen models; the larger profiles use 8 and 50. These are models fit using the same training split, not independent new datasets.

In [ ]:
generalization_results = runtime.run_section(
    "generalization", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
)
display(runtime.compact_summary(generalization_results))

## 7. Homogeneous neural networks
Compare methods for independently trained networks with the same (50,100) hidden-layer widths. Quick uses three networks and 2,000 synthetic observations; larger profiles use five networks and 20,000 synthetic observations. The optional MIQCP reference is a minimum-L1 classification-feasibility comparator under the common L2 budget.

In [ ]:
neural_results = runtime.run_section(
    "nonlinear_baselines", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
)
display(runtime.compact_summary(neural_results))

## 8. Budget, validity, and competing-set size
Train the mixed competing sets (2,2,2), (4,4,4), and (8,8,8). Save budget-versus-validity curves for each set and set-size-versus-validity curves for each budget, separately by dataset. The models are nested across set sizes and the factual cohorts are shared. This cell generates the inputs needed by the next experiment.

In [ ]:
tradeoff_results = runtime.run_section(
    "cost_validity", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
)
display(runtime.compact_summary(tradeoff_results))

## 9. Burden of multiplicity — differentiable models
Use the model checkpoints, factuals, budgets, and ORPM points generated in Section 8. Search for recourse against each model individually. The primary statistic is the worst individual-model validity minus joint validity, with the minimum taken **within each seed** and confidence intervals computed on paired seed differences. Also report the fraction of individuals with a valid singleton witness for every model. This is achieved-search burden, not an intrinsic infeasibility certificate or an extra monetary cost.

In [ ]:
if "tradeoff_results" not in globals():
    raise RuntimeError("Run Section 8 first; no previous results are required.")
multiplicity_results = runtime.run_burden(
    tradeoff_results, RUN_MODE, RESUME, SHOW_PLOTS,
    name="multiplicity_burden_differentiable",
)
display(multiplicity_results)


## 10. Tree loss and duality-gap convergence
Use original decision-tree predictions and the mixed-integer oracle. Report the best feasible point and uniform-mixture gap separately; averaging feature vectors is not a valid tree convergence argument.

In [ ]:
if RUN_TREES:
    tree_gap_results = runtime.run_section(
        "tree_convergence", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
    )
    display(runtime.compact_summary(tree_gap_results))
else:
    tree_gap_results = None
    print("Tree section explicitly disabled by RUN_TREES=False.")

## 11. Tree and random-forest baselines
Run decision trees, probability-averaging forests, and hard-voting forests. Compare ORPM-tree with Joint-minimax, ADV/ROAR surrogates, constrained RobX, and OCEAN-style recourse. The matched hard-voting experiment also includes the DirectSAA and RobustSAA formulations. Adaptations retain explicit labels.

In [ ]:
if RUN_TREES:
    tree_comparison_results = runtime.run_section(
        "tree_comparison", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
    )
    display(runtime.compact_summary(tree_comparison_results))
else:
    tree_comparison_results = None
    print("Tree section explicitly disabled by RUN_TREES=False.")

## 12. Tree budget–validity trade-offs
Reuse each trained tree set and factual cohort across its budget grid. Save a separate figure and table for every dataset and model family. This supplies the inputs to the tree multiplicity analysis below.

In [ ]:
if RUN_TREES:
    tree_tradeoff_results = runtime.run_section(
        "tree_tradeoff", RUN_MODE, SEED, RESUME, show=SHOW_PLOTS,
    )
    display(runtime.compact_summary(tree_tradeoff_results))
else:
    tree_tradeoff_results = None
    print("Tree section explicitly disabled by RUN_TREES=False.")

## 13. Burden of multiplicity — tree models
Use the fresh tree trade-offs from Section 12. Individual-model searches use the original tree predictor and retain solver status and bound information. As in the differentiable experiment, valid saved joint witnesses are reused as singleton witnesses, and unresolved searches remain distinguishable from certified infeasibility.

In [ ]:
if RUN_TREES:
    if tree_tradeoff_results is None:
        raise RuntimeError("Run Section 12 before this cell.")
    tree_multiplicity_results = runtime.run_burden(
        tree_tradeoff_results, RUN_MODE, RESUME, SHOW_PLOTS,
        name="multiplicity_burden_tree",
    )
    display(tree_multiplicity_results)
else:
    tree_multiplicity_results = None
    print("Tree burden analysis explicitly disabled by RUN_TREES=False.")


## 14. Output index and interpretation
All results are under `results/v6/<mode>/<experiment>/<timestamp-and-id>/`. Figures and compact tables are nested by dataset in `figures/<dataset>/` and `table/<dataset>/` within the corresponding experiment directory. Exact paths are listed below.

Quick results verify operation on a small workload. Two seeds and three instances do not support paper-level precision. The reported confidence intervals use equal-weight seed means; zero observed between-seed variance is flagged rather than described as certainty. Gaps in feasibility or solver status remain visible in the per-instance records. All feature-domain experiments use a continuous relaxation; historical and categorical-coordinate changes are not causal recommendations.

To run more cases, change **only `RUN_MODE`** and rerun from the control cell onward (or Restart and Run All). Data and code are shared, but results are separated by mode. The documented full profile reproduces the recorded target convention; resolve the German/Polish target audit and rerun before making favorable-recourse claims for those datasets.

In [ ]:
display(runtime.run_index())
print("Results root:", ROOT / "results" / "v6" / campaign.mode_name(RUN_MODE))
